In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.base import BaseEstimator, TransformerMixin
from lightgbm import LGBMClassifier
import os
import sklearn

In [2]:
sklearn.set_config(transform_output="pandas")

In [3]:
# Load data - using train_original.csv this time!
train = pd.read_csv('../datasets/train_original.csv')
test = pd.read_csv('../datasets/test.csv')

In [4]:
X = train.drop(['id', 'addicted_label'], axis=1)
y = train['addicted_label']
X_test = test.drop(['id'], axis=1)

In [5]:
# Proper Ordinal Mappings
stress_mapping = {'Low': 0, 'Medium': 1, 'High': 2, 'Unknown': -1}
impact_mapping = {'No': 0, 'Yes': 1, 'Unknown': -1}

In [6]:
def apply_mappings(df):
    df_out = df.copy()
    # Map ordinals, fill missing in these specific columns with 'Unknown' mapping (-1) if any
    df_out['stress_level'] = df_out['stress_level'].map(stress_mapping).fillna(-1).astype(int)
    df_out['academic_work_impact'] = df_out['academic_work_impact'].map(impact_mapping).fillna(-1).astype(int)
    
    # Leave gender as category
    df_out['gender'] = df_out['gender'].fillna('Unknown').astype('category')
    
    return df_out

In [7]:
X_preprocessed = apply_mappings(X)
X_test_preprocessed = apply_mappings(X_test)

In [8]:
# Feature Engineering
denom_screen = X_preprocessed['daily_screen_time_hours'].replace(0, 0.001)
denom_notif = X_preprocessed['notifications_per_day'].replace(0, 0.001)
X_preprocessed['social_media_ratio'] = X_preprocessed['social_media_hours'] / denom_screen
X_preprocessed['gaming_ratio'] = X_preprocessed['gaming_hours'] / denom_screen
X_preprocessed['work_study_ratio'] = X_preprocessed['work_study_hours'] / denom_screen
X_preprocessed['app_opens_per_hour'] = X_preprocessed['app_opens_per_day'] / denom_screen
X_preprocessed['notifications_to_opens_ratio'] = X_preprocessed['app_opens_per_day'] / denom_notif
X_preprocessed['sleep_deficit'] = 8.0 - X_preprocessed['sleep_hours']

In [9]:
denom_screen_test = X_test_preprocessed['daily_screen_time_hours'].replace(0, 0.001)
denom_notif_test = X_test_preprocessed['notifications_per_day'].replace(0, 0.001)
X_test_preprocessed['social_media_ratio'] = X_test_preprocessed['social_media_hours'] / denom_screen_test
X_test_preprocessed['gaming_ratio'] = X_test_preprocessed['gaming_hours'] / denom_screen_test
X_test_preprocessed['work_study_ratio'] = X_test_preprocessed['work_study_hours'] / denom_screen_test
X_test_preprocessed['app_opens_per_hour'] = X_test_preprocessed['app_opens_per_day'] / denom_screen_test
X_test_preprocessed['notifications_to_opens_ratio'] = X_test_preprocessed['app_opens_per_day'] / denom_notif_test
X_test_preprocessed['sleep_deficit'] = 8.0 - X_test_preprocessed['sleep_hours']

In [10]:
print("Training model...")
# Setting scale_pos_weight for imbalance, and letting LGBM handle NaNs natively for numericals
# Ratio is 200k / 490k ~ 0.41, but actually pos_weight = neg / pos = 200k / 490k = 0.41 
# Wait, if 1 is addicted, then it's the majority class! So it's 200k (class 0) / 490k (class 1) ~ 0.41.
# Usually is_unbalance=True handles it automatically.
model = LGBMClassifier(n_estimators=300, learning_rate=0.05, random_state=42, verbose=-1, is_unbalance=True)
model.fit(X_preprocessed, y)

Training model...


,learning_rate,0.05
,n_estimators,300
,random_state,42
,verbose,-1
,is_unbalance,True
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,subsample_for_bin,200000
,objective,None
,class_weight,None


In [11]:
print("Predicting final results...")
final_preds = model.predict_proba(X_test_preprocessed)[:, 1]

Predicting final results...


In [12]:
os.makedirs('../submissions', exist_ok=True)
submission = pd.DataFrame({'id': test['id'], 'addicted_label': final_preds})
submission.to_csv('../submissions/native_nan_fixed.csv', index=False)
print("Submission saved to submissions/native_nan_fixed.csv")

Submission saved to submissions/native_nan_fixed.csv
